In [10]:
import numpy as np
import pandas as pd
import io
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import cm
from matplotlib import colorbar
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, fisher_exact, describe
from scipy.spatial.distance import jaccard, euclidean, cosine, pdist
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.cluster import MeanShift
from scipy.cluster import hierarchy
import json
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


today = date.today()

In [2]:
def kde_wrapper(data, bw=0.1):
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    
    sns.kdeplot(data, bw_adjust=bw)
    print(describe(data))
    return fig

**Functions**

In [2]:
# def combine_scores_multiple(DHS_scaled_vals, DHS_similar_regions, TF_scaled_vals, TF_similar_regions, epi_scaled_vals,
#                             epi_similar_regions, coex_scaled_vals, coex_similar_regions, cooccur_scaled_vals, cooccur_similar_regions,
#                             database_scaled_vals, database_similar_regions, exper_scaled_vals, exper_similar_regions,fusion_scaled_vals, fusion_similar_regions,
#                             homology_scaled_vals, homology_similar_regions, neigh_scaled_vals, neigh_similar_regions, textmin_scaled_vals, textmin_similar_regions):
#     #max ind for combined score array
#     max_ind = np.max([DHS_similar_regions, TF_similar_regions, epi_similar_regions, coex_similar_regions,
#                      cooccur_similar_regions, database_similar_regions, exper_similar_regions, fusion_similar_regions, homology_similar_regions,
#                      neigh_similar_regions, textmin_similar_regions])

#     #DHS scores
#     DHS_arr = create_arr_multiple(DHS_similar_regions, DHS_scaled_vals, max_ind)
#     #epilogos scores
#     epi_arr = create_arr_multiple(epi_similar_regions, epi_scaled_vals, max_ind)
#     #TF motif scores
#     TF_arr = create_arr_multiple(TF_similar_regions, TF_scaled_vals, max_ind)
#     #coexpression scores
#     coex_arr = create_arr_multiple(coex_similar_regions, coex_scaled_vals, max_ind)
#     #cooccurrence scores
#     cooccur_arr = create_arr_multiple(cooccur_similar_regions, cooccur_scaled_vals, max_ind)
#     #database scores
#     database_arr = create_arr_multiple(database_similar_regions, database_scaled_vals, max_ind)
#     #experiment scores
#     exper_arr = create_arr_multiple(exper_similar_regions, exper_scaled_vals, max_ind)
#     #fusion scores
#     fusion_arr = create_arr_multiple(fusion_similar_regions, fusion_scaled_vals, max_ind)
#     #homology scores
#     homology_arr = create_arr_multiple(homology_similar_regions, homology_scaled_vals, max_ind)
#     #neighborhood scores
#     neigh_arr = create_arr_multiple(neigh_similar_regions, neigh_scaled_vals, max_ind)
#     #text mining scores
#     textmin_arr = create_arr_multiple(textmin_similar_regions, textmin_scaled_vals, max_ind)

    
#     #fuse STRING scores first
#     string_arr = 1 - ((1 - coex_arr) * (1 - cooccur_arr) * (1 - database_arr) * (1 - exper_arr) *
#                       (1 - fusion_arr) * (1 - homology_arr) * (1 - neigh_arr) * (1 - textmin_arr))
    
    
#     #scoring all metrics
#     score_arr = 1 - ((1 - epi_arr) * (1 - DHS_arr) * (1 - TF_arr) * (1 - coex_arr) * (1 - cooccur_arr) * (1 - database_arr) * (1 - exper_arr) *
#                      (1 - fusion_arr) * (1 - homology_arr) * (1 - neigh_arr) * (1 - textmin_arr))

#     #labeling most contributing metrics for each region
#     contributing_metrics_arr = np.argmax(np.array((DHS_arr, TF_arr, epi_arr, coex_arr, cooccur_arr, database_arr, exper_arr, fusion_arr,
#                                                   homology_arr, neigh_arr, textmin_arr)), axis=0)

#     return (score_arr, contributing_metrics_arr, max_ind, DHS_arr, TF_arr, epi_arr, coex_arr, 
#             cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, neigh_arr, textmin_arr, string_arr)

def combine_scores_multiple(DHS_scaled_vals, DHS_similar_regions, TF_scaled_vals, TF_similar_regions, epi_scaled_vals,
                            epi_similar_regions, coex_scaled_vals, coex_similar_regions, cooccur_scaled_vals, cooccur_similar_regions,
                            database_scaled_vals, database_similar_regions, exper_scaled_vals, exper_similar_regions,fusion_scaled_vals, fusion_similar_regions,
                            homology_scaled_vals, homology_similar_regions, neigh_scaled_vals, neigh_similar_regions, textmin_scaled_vals, textmin_similar_regions):
    #max ind for combined score array
    max_ind = np.max([DHS_similar_regions, TF_similar_regions, epi_similar_regions, coex_similar_regions,
                     cooccur_similar_regions, database_similar_regions, exper_similar_regions, fusion_similar_regions, homology_similar_regions,
                     neigh_similar_regions, textmin_similar_regions])

    #DHS scores
    DHS_arr = create_arr_multiple(DHS_similar_regions, DHS_scaled_vals, max_ind)
    #epilogos scores
    epi_arr = create_arr_multiple(epi_similar_regions, epi_scaled_vals, max_ind)
    #TF motif scores
    TF_arr = create_arr_multiple(TF_similar_regions, TF_scaled_vals, max_ind)
    #coexpression scores
    coex_arr = create_arr_multiple(coex_similar_regions, coex_scaled_vals, max_ind)
    #cooccurrence scores
    cooccur_arr = create_arr_multiple(cooccur_similar_regions, cooccur_scaled_vals, max_ind)
    #database scores
    database_arr = create_arr_multiple(database_similar_regions, database_scaled_vals, max_ind)
    #experiment scores
    exper_arr = create_arr_multiple(exper_similar_regions, exper_scaled_vals, max_ind)
    #fusion scores
    fusion_arr = create_arr_multiple(fusion_similar_regions, fusion_scaled_vals, max_ind)
    #homology scores
    homology_arr = create_arr_multiple(homology_similar_regions, homology_scaled_vals, max_ind)
    #neighborhood scores
    neigh_arr = create_arr_multiple(neigh_similar_regions, neigh_scaled_vals, max_ind)
    #text mining scores
    textmin_arr = create_arr_multiple(textmin_similar_regions, textmin_scaled_vals, max_ind)

    
    #fuse STRING scores first
    string_arr = 1 - ((1 - coex_arr) * (1 - cooccur_arr) * (1 - database_arr) * (1 - exper_arr) *
                      (1 - fusion_arr) * (1 - homology_arr) * (1 - neigh_arr) * (1 - textmin_arr))
    
    
    #scoring all metrics
    chromatin_arr = 1 - ((1 - epi_arr) * (1 - DHS_arr) * (1 - TF_arr))

    #labeling most contributing metrics for each region
    contributing_chromatin_metrics_arr = np.argmax(np.array((DHS_arr, TF_arr, epi_arr)), axis=0)
    contributing_string_metrics_arr = np.argmax(np.array((coex_arr, cooccur_arr, database_arr,
                                                          exper_arr, fusion_arr, homology_arr,
                                                          neigh_arr, textmin_arr)), axis=0)

    return (chromatin_arr, string_arr, contributing_chromatin_metrics_arr, contributing_string_metrics_arr, max_ind, DHS_arr, 
            TF_arr, epi_arr, coex_arr, cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, 
            neigh_arr, textmin_arr)

def region_return_multiple(score_arr, roi):

    #top 100 region inds and scores
    rec_inds = np.argsort(-score_arr)[:, :100]
    top_region_inds = np.hstack((roi.reshape(-1, 1), rec_inds))
    # top_region_scores = np.append(1, score_arr[rec_inds])

    return top_region_inds

def roi_url(roi, top_regions):
    today = date.today()
    roi_url_name = 'region_comparison_recommended_regions_roi_'+str(roi)+'_scale_25kb_CT' + today.strftime("%Y%m%d") + '.bed'
    bed_path_name = os.path.join('/home/ctrader/public_html/', roi_url_name)
    np.savetxt(bed_path_name, top_regions, delimiter='\t', fmt='%s')
    link_path1 = 'https://epilogos.altius.org:1234/?application=viewer&sampleSet=vC&mode=single&genome=hg38&model=18&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2F' + roi_url_name
    link_path2 = 'https://index.altius.org:1234/?application=viewer&mode=pd&chrLeft=chr11&chrRight=chr11&start=47329800&stop=47356800&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2F' + roi_url_name
    return link_path1, link_path2, bed_path_name

def load_hdf5_files_multiple(hdf5_path, roi):
    #take in a file path and load scaled scores and similar regions
    hdf5_file = h5py.File(hdf5_path, 'r')
    similar_regions = hdf5_file['top_1000_similar_regions'][roi]
    factor_ranks = hdf5_file['ranked_regions'][roi]
    sigmoid_scaled_vals = hdf5_file['top_1000_sigmoid_scaled_vals'][roi]
    hdf5_file.close()
    return sigmoid_scaled_vals, similar_regions, factor_ranks

def load_STRING_hdf5_files_multiple(hdf5_path, roi):
    #load in tracking genes
    hdf5_file = h5py.File(hdf5_path, 'r')
    ori_genes = np.char.decode(hdf5_file['top_1000_original_genes'][roi], 'utf-8')
    dest_genes = np.char.decode(hdf5_file['top_1000_destination_genes'][roi], 'utf-8')
    similar_regions = hdf5_file['top_1000_similar_regions'][roi]
    scaled_vals = hdf5_file['top_1000_scaled_vals'][roi]
#     STRING_vals = hdf5_file['top_1000_STRING_vals'][roi]
    hdf5_file.close()
    return scaled_vals, similar_regions, ori_genes, dest_genes

def create_arr_multiple(similar_regions, scaled_vals, max_ind):
    #empty array
    arr = np.zeros((len(similar_regions), max_ind + 1))
    
    #find the values to change
    cols = similar_regions
    rows = np.array([np.arange(len(similar_regions))]*len(similar_regions[0])).T
    
    #set values
    arr[rows, cols] = scaled_vals
    return arr

def load_hdf5_genes_multiple(ori_genes, dest_genes, similar_regions, max_ind):
    #establishing gene arrays
    ori_arr = np.where(np.zeros((len(similar_regions), max_ind + 1))==0, None, None)
    dest_arr = np.where(np.zeros((len(similar_regions), max_ind + 1))==0, None, None)

    #filling gene arrays
    #find the values to change
    cols = similar_regions
    rows = np.array([np.arange(len(similar_regions))]*len(similar_regions[0])).T
#     print(cols)
#     print(rows)
#     print(ori_genes.shape)

    #set values
    ori_arr[rows, cols] = ori_genes
    dest_arr[rows, cols] = dest_genes
    return np.array([ori_arr, dest_arr])

def get_factor_order(path):
    if path[-5:]=='.hdf5':
        hdf5_file = h5py.File(path, 'r')
        factor_order = np.char.decode(hdf5_file['comp_order'][:])
        hdf5_file.close()
    else:
        f = open(path, 'r')
        factor_order = np.array(f.readline().strip().split('\t'))[3:]
        f.close()
    return factor_order

def load_rank_path(hdf5_path, roi):
    hdf5_file = h5py.File(hdf5_path, 'r')
    rank_arr = hdf5_file['ranked_regions'][roi]
    max_ranks = hdf5_file['max_ranks'][:]
    hdf5_file.close()
    return rank_arr, max_ranks

**Data**

In [31]:
DHS_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.DHSsigmoideuc.hdf5'
TF_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.TFsigmoideuc.hdf5'
epilogos_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.epilogossigmoideuc.hdf5'
coex_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.coexpression_human_transferred_maxSTRINGsimilarity.hdf5'
cooccur_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.cooccurenceSTRINGsimilarity.hdf5'
database_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.database_human_transferred_maxSTRINGsimilarity.hdf5'
exper_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.experiments_human_transferred_maxSTRINGsimilarity.hdf5'
fusion_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.fusionSTRINGsimilarity.hdf5'
homology_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.homologySTRINGsimilarity.hdf5'
neigh_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.neighborhood_human_transferred_maxSTRINGsimilarity.hdf5'
textmin_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.textmining_human_transferred_maxSTRINGsimilarity.hdf5'
# STRING_ranks_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221109.STRINGsimilarityrankedregions.hdf5'
STRING_ranks_hdf5_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221115.top_gene.combined_scores_above_400.counts_by_metric.hdf5'

DHS_mapped_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.DHSmapped.bed'
TF_mapped_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.TFmapped.bed'
epilogos_mapped_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.epilogosmapped.bed'

regions_path='/home/ctrader/region_similarity/data/Altius_25kb_hg38_maxMean_CT20220815.sorted.c1t3.bed'
gz_file_path='/home/ctrader/region_similarity/data/Altius_25kb_hg38_maxMean_CT20220815.sorted.c1t3.rownum4.bed.gz'

geneID2name_path='/home/ctrader/region_similarity/data/gene_proteinID2genename_dict_CT20220712.pickle'

DHS_base_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.DHScounts.hdf5'
TF_base_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.TFcounts.hdf5'
epi_base_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.epilogosmeanmax.hdf5'

In [32]:
#loading order of each factor
DHS_factor_order = get_factor_order(DHS_mapped_path)
TF_factor_order = get_factor_order(TF_mapped_path)
epilogos_factor_order = get_factor_order(epilogos_mapped_path)

In [33]:
pickle_in = open(geneID2name_path, 'rb')
geneID2name = pickle.load(pickle_in)
pickle_in.close()

In [34]:
genomic_location = 'chr1:198636000-198661000'
# genomic_location = 'chr1:198636000-198636001'
gz_tb = tabix.open(gz_file_path)
# roi = int(list(gz_tb.querys(genomic_location))[0][3])
roi = np.array(list(gz_tb.querys(genomic_location)))[:, 3].astype(int)
print(roi)

[6434 6435]


In [35]:
roi = np.array([78852])

In [36]:
f = open(regions_path, 'r')
regions = f.readlines()
f.close()

regions[78852].strip().split('\t')

['chr7', '50298000', '50323000']

In [37]:
t = time.time()

###Load rank and ind arrays
#DHS counts
DHS_scaled_vals, DHS_similar_regions, DHS_roi_ranks = load_hdf5_files_multiple(DHS_hdf5_path, roi)
#TF motif counts
TF_scaled_vals, TF_similar_regions, TF_roi_ranks = load_hdf5_files_multiple(TF_hdf5_path, roi)
#epilogos score sum and max
epi_scaled_vals, epi_similar_regions, epi_roi_ranks = load_hdf5_files_multiple(epilogos_hdf5_path, roi)
#coexpression STRING scores
coex_scaled_vals, coex_similar_regions, coex_ori, coex_dest = load_STRING_hdf5_files_multiple(coex_hdf5_path, roi)
#cooccurence STRING scores
cooccur_scaled_vals, cooccur_similar_regions, cooccur_ori, cooccur_dest = load_STRING_hdf5_files_multiple(cooccur_hdf5_path, roi)
#database STRING scores
database_scaled_vals, database_similar_regions, database_ori, database_dest = load_STRING_hdf5_files_multiple(database_hdf5_path, roi)
#experiments STRING scores
exper_scaled_vals, exper_similar_regions, exper_ori, exper_dest = load_STRING_hdf5_files_multiple(exper_hdf5_path, roi)
#fusion STRING scores
fusion_scaled_vals, fusion_similar_regions, fusion_ori, fusion_dest = load_STRING_hdf5_files_multiple(fusion_hdf5_path, roi)
#homology STRING scores
homology_scaled_vals, homology_similar_regions, homology_ori, homology_dest = load_STRING_hdf5_files_multiple(homology_hdf5_path, roi)
#neighborhood STRING scores
neigh_scaled_vals, neigh_similar_regions, neigh_ori, neigh_dest = load_STRING_hdf5_files_multiple(neigh_hdf5_path, roi)
#textmining STRING scores
textmin_scaled_vals, textmin_similar_regions, textmin_ori, textmin_dest = load_STRING_hdf5_files_multiple(textmin_hdf5_path, roi)

###Combine scores 
(chromatin_arr, string_arr, 
 contributing_chromatin_metrics_arr, 
 contributing_string_metrics_arr, 
 max_ind, DHS_arr, TF_arr, 
 epi_arr, coex_arr, cooccur_arr, 
 database_arr, exper_arr, fusion_arr, 
 homology_arr, neigh_arr, textmin_arr) = combine_scores_multiple(DHS_scaled_vals, DHS_similar_regions, 
                                                                 TF_scaled_vals, TF_similar_regions, 
                                                                 epi_scaled_vals, epi_similar_regions, 
                                                                 coex_scaled_vals, coex_similar_regions, 
                                                                 cooccur_scaled_vals, cooccur_similar_regions,
                                                                 database_scaled_vals, database_similar_regions, 
                                                                 exper_scaled_vals, exper_similar_regions, 
                                                                 fusion_scaled_vals, fusion_similar_regions,
                                                                 homology_scaled_vals, homology_similar_regions, 
                                                                 neigh_scaled_vals, neigh_similar_regions, 
                                                                 textmin_scaled_vals, textmin_similar_regions)

#load in gene names for STRING scores
coex_genes = load_hdf5_genes_multiple(coex_ori, coex_dest, coex_similar_regions, max_ind)
cooccur_genes = load_hdf5_genes_multiple(cooccur_ori, cooccur_dest, cooccur_similar_regions, max_ind)
database_genes = load_hdf5_genes_multiple(database_ori, database_dest, database_similar_regions, max_ind)
exper_genes = load_hdf5_genes_multiple(exper_ori, exper_dest, exper_similar_regions, max_ind)
fusion_genes = load_hdf5_genes_multiple(fusion_ori, fusion_dest, fusion_similar_regions, max_ind)
homology_genes = load_hdf5_genes_multiple(homology_ori, homology_dest, homology_similar_regions, max_ind)
neigh_genes = load_hdf5_genes_multiple(neigh_ori, neigh_dest, neigh_similar_regions, max_ind)
textmin_genes = load_hdf5_genes_multiple(textmin_ori, textmin_dest, textmin_similar_regions, max_ind)

###Return regions
top_region_inds = region_return_multiple(chromatin_arr, roi)

# top_region_inds = region_return_multiple(string_arr, roi)

print(time.time()-t)

3.3864259719848633


In [120]:
# #values for top 100 regions
# cols = top_region_inds[:, 1:]
# rows = np.array([np.arange(len(top_region_inds[:, 1:]))]*len(top_region_inds[:, 1:][0])).T

# # DHS_arr_top_100 = DHS_arr[rows, cols]
# # TF_arr_top_100 = TF_arr[rows, cols]
# # epi_arr_top_100 = epi_arr[rows, cols]

# chromatin_arr_top_100 = np.array((DHS_arr, TF_arr, epi_arr))[:, rows, cols]
# chromatin_metric_order = ['DHS', 'TF', 'epilogos']

# string_arr_top_100 = np.array((coex_arr, cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, 
#                                neigh_arr, textmin_arr))[:, rows, cols]
# string_metric_order = ['Co-expression', 'Co-occurence', 'Co-annotation', 
#                        'Interactions in vitro', 'Fusion in Other Species', 'Common Ancestry', 
#                        'being Nearby', 'Co-citation']
# string_colors = np.array(['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c', '#fb9a99', '#e31a1c', '#fdbf6f', '#ff7f00', 
#                           '#cab2d6', '#6a3d9a', '#ffff99'])

In [121]:
# #genomic coordinates for each region
# f = open(regions_path, 'r')
# regions = []
# for line in f.readlines():
#     line = line.strip().split('\t')
#     regions.append(line[0] + ':' + line[1] + '-' + line[2])
# f.close()

# top_region_coords = np.array(regions)[top_region_inds[:, 1:]]

In [141]:
# #old can delete
# chromatin_return = []
# for region in range(100):
#     region_dict = {}
#     for metric, name in zip(range(len(chromatin_arr_top_100[:, 0])), chromatin_metric_order):
#         region_dict[name] = chromatin_arr_top_100[metric, 0, region]
#     region_dict['region'] = region
#     region_dict['coordinates'] = top_region_coords[0, region]
    
#     chromatin_return.append(region_dict)

# json_return = json.dumps(chromatin_return)
# sys.stdout.write(json_return)
# x = chromatin_return

In [123]:
#values for top 100 regions (including query region)
cols = top_region_inds
rows = np.array([np.arange(len(top_region_inds))]*len(top_region_inds[0])).T

chromatin_arr_top_100 = np.array((DHS_arr, TF_arr, epi_arr))[:, rows, cols]
chromatin_metric_order = ['DHS', 'TF', 'epilogos']

string_arr_top_100 = np.array((coex_arr, cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, 
                               neigh_arr, textmin_arr))[:, rows, cols]
string_metric_order = ['Co-expression', 'Co-occurence', 'Co-annotation', 
                       'Interactions in vitro', 'Fusion in Other Species', 'Common Ancestry', 
                       'being Nearby', 'Co-citation']
string_colors = np.array(['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c', '#fb9a99', '#e31a1c', '#fdbf6f', '#ff7f00', 
                          '#cab2d6', '#6a3d9a', '#ffff99'])

In [149]:
sort_order = np.argsort(top_region_inds[0])

In [159]:
x = top_region_inds[0][sort_order]
x

array([  112,  1103,  3677,  5458,  6520,  6585,  6645,  6693,  9462,
        9587, 11063, 11873, 12594, 12801, 12833, 14297, 14604, 15717,
       15920, 15923, 15970, 15995, 16090, 16138, 18125, 18211, 18224,
       18259, 19854, 20006, 20058, 20235, 24324, 25255, 25286, 27255,
       27733, 30082, 30602, 33883, 33952, 34033, 34265, 34664, 35737,
       35744, 36458, 36700, 36754, 37076, 37130, 39251, 39560, 40669,
       41718, 43076, 43527, 45499, 46226, 47651, 48219, 48846, 50635,
       52256, 52359, 53785, 55922, 58107, 59929, 61853, 62812, 66318,
       67986, 70454, 70737, 70908, 71274, 71788, 72145, 72741, 72987,
       72994, 73282, 74795, 75295, 76665, 77203, 78220, 78443, 78480,
       78852, 81319, 81691, 81788, 82037, 85554, 86401, 90058, 91037,
       92138, 96216])

In [157]:
top_region_inds[0]

array([78852, 81788, 20006, 11063, 34664,  9462, 96216, 37130, 16090,
       39560, 43527, 25255, 34265, 48846,  6520, 67986, 75295, 37076,
       72994, 70908, 11873,  9587, 12833,  1103, 61853, 70737, 33952,
       78480,  6645, 62812,  3677, 12594, 27255, 90058, 18211, 27733,
       73282, 45499, 58107,  6585, 78443, 72145, 81691, 20235, 16138,
       35744, 15923, 48219, 43076, 92138, 85554,  5458, 86401, 18224,
       36458, 72987, 20058, 40669, 15920,   112, 33883,  6693, 18259,
       36700, 52256, 34033, 47651, 41718, 35737, 14604, 12801, 39251,
       15717, 30602, 18125, 14297, 30082, 78220, 71274, 46226, 50635,
       53785, 82037, 81319, 25286, 15970, 74795, 52359, 55922, 77203,
       36754, 72741, 70454, 15995, 24324, 91037, 66318, 71788, 59929,
       76665, 19854])

In [160]:
x[np.argsort(sort_order)]

array([78852, 81788, 20006, 11063, 34664,  9462, 96216, 37130, 16090,
       39560, 43527, 25255, 34265, 48846,  6520, 67986, 75295, 37076,
       72994, 70908, 11873,  9587, 12833,  1103, 61853, 70737, 33952,
       78480,  6645, 62812,  3677, 12594, 27255, 90058, 18211, 27733,
       73282, 45499, 58107,  6585, 78443, 72145, 81691, 20235, 16138,
       35744, 15923, 48219, 43076, 92138, 85554,  5458, 86401, 18224,
       36458, 72987, 20058, 40669, 15920,   112, 33883,  6693, 18259,
       36700, 52256, 34033, 47651, 41718, 35737, 14604, 12801, 39251,
       15717, 30602, 18125, 14297, 30082, 78220, 71274, 46226, 50635,
       53785, 82037, 81319, 25286, 15970, 74795, 52359, 55922, 77203,
       36754, 72741, 70454, 15995, 24324, 91037, 66318, 71788, 59929,
       76665, 19854])

In [124]:
#genomic coordinates for each region
f = open(regions_path, 'r')
regions = []
for line in f.readlines():
    line = line.strip().split('\t')
    regions.append(line[0] + ':' + line[1] + '-' + line[2])
f.close()

top_region_coords = np.array(regions)[top_region_inds]

In [125]:
#new
chromatin_return = []
for rank in range(101):
    region_dict = {}
    for metric, name in zip(range(len(chromatin_metric_order)), chromatin_metric_order):
        region_dict[name] = chromatin_arr_top_100[metric, 0, rank]
    region_dict['rank'] = rank
    region_dict['coordinates'] = top_region_coords[0, rank]
    region_dict['region_ind'] = int(top_region_inds[0, rank])
    chromatin_return.append(region_dict)

json_return = json.dumps(chromatin_return)
sys.stdout.write(json_return)
y = chromatin_return

[{"DHS": 0.0, "TF": 0.0, "epilogos": 0.0, "rank": 0, "coordinates": "chr7:50298000-50323000", "region_ind": 78852}, {"DHS": 0.6933488840091147, "TF": 0.0, "epilogos": 0.831488302585041, "rank": 1, "coordinates": "chr7:142796600-142821600", "region_ind": 81788}, {"DHS": 0.7428823178062915, "TF": 0.0, "epilogos": 0.7983340387767834, "rank": 2, "coordinates": "chr12:113054400-113079400", "region_ind": 20006}, {"DHS": 0.7956223025189643, "TF": 0.0, "epilogos": 0.7022727037189326, "rank": 3, "coordinates": "chr10:96716400-96741400", "region_ind": 11063}, {"DHS": 0.7034783482908946, "TF": 0.0, "epilogos": 0.7861314217610373, "rank": 4, "coordinates": "chr17:16475600-16500600", "region_ind": 34664}, {"DHS": 0.6868243525978138, "TF": 0.0, "epilogos": 0.759574017506481, "rank": 5, "coordinates": "chr10:45356400-45381400", "region_ind": 9462}, {"DHS": 0.0, "TF": 0.5496685235320857, "epilogos": 0.8292572729488616, "rank": 6, "coordinates": "chrX:151962000-151987000", "region_ind": 96216}, {"DHS":

**Regional Sentences**

In [137]:
roi = np.arange(98629)
roi = np.arange(101)*500

In [139]:
roi = np.array([4143])

In [140]:
#load rank arrays
DHS_rank_arr, DHS_max_ranks = load_rank_path(DHS_hdf5_path, roi)
TF_rank_arr, TF_max_ranks = load_rank_path(TF_hdf5_path, roi)
epi_rank_arr, epi_max_ranks = load_rank_path(epilogos_hdf5_path, roi)
STRING_rank_arr, STRING_max_ranks = load_rank_path(STRING_ranks_hdf5_path, roi)
STRING_rank_arr = STRING_rank_arr[:, -1].reshape(-1, 1)
STRING_max_ranks = STRING_max_ranks[-1]

#load factor orders
DHS_factor_order = get_factor_order(DHS_mapped_path)
TF_factor_order = get_factor_order(TF_mapped_path)
epi_factor_order = get_factor_order(epilogos_mapped_path)
STRING_factor_order = get_factor_order(STRING_ranks_hdf5_path)[[-1]]

In [17]:
    epi_conversion_dict = {
        'EnhA1': 'active enhancer 1',
        'EnhA2': 'active enhancer 2',
        'EnhBiv': 'bivalent enhancer', 
        'EnhG1': 'genic enhancer 1',
        'EnhG2': 'genic enhancer 2', 
        'EnhWk': 'weak enhancer', 
        'Het': 'heterochromatin', 
        'Quies': 'quiescent',
        'ReprPC': 'repressed polycomb', 
        'ReprPCWk': 'weak repressed polycomb',
        'TssA': 'active TSS',
        'TssBiv': 'bivalent TSS',
        'TssFlnk': 'flanking TSS',
        'TssFlnkD': 'downstream flanking TSS',
        'TssFlnkU': 'upstream flanking TSS',
        'Tx': 'strong transcription',
        'TxWk': 'weak transcription',
        'ZNF_Rpts': 'ZNF genes + repeats'
    }

    DHS_conversion_dict = {
        'Cancer_epithelial': 'cancer/epithelial',
        'Cardiac': 'cardiac',
        'Digestive': 'digestive',
        'Lymphoid': 'lymphoid',
        'Musculoskeletal': 'musculoskeletal',
        'Myeloid_erythroid': 'myeloid/erythroid',
        'Neural': 'neural',
        'Organdevel_renal': 'organ development/renal',
        'Placental_trophoblast': 'placental/trophoblast',
        'Primitive_embryonic': 'primitive/embryonic',
        'Pulmonarydevel': 'pulmonary development',
        'Renal_cancer': 'renal/cancer',
        'StromalA': 'stromal A',
        'StromalB': 'stromal B',
        'Tissueinvariant': 'tissue invariant',
        'Vascular_endothelial': 'vascular/endothelial'
    }

    TF_conversion_dict = {}
    for factor in TF_factor_order:
        TF_conversion_dict[factor] = factor

    STRING_conversion_dict = {
        'coexpression' : 'co-expression',
        'cooccurence' : 'co-evolution',
        'database' : 'co-annotation',
        'experiments' : 'in vitro',
        'fusion' : 'non-human fused',
        'homology' : 'common ancestry',
        'neighborhood' : 'neighboring',
        'textmining' : 'literature based',
        'combined_score': 'combined'
    }

In [18]:
    rank_arrs = [DHS_rank_arr, TF_rank_arr, epi_rank_arr[:, :18], epi_rank_arr[:, 18:], STRING_rank_arr]
    factor_arrs = [DHS_factor_order, TF_factor_order, epi_factor_order, epi_factor_order, STRING_factor_order]
    max_rank_arrs = [DHS_max_ranks, TF_max_ranks, epi_max_ranks[:18], epi_max_ranks[18:], STRING_max_ranks]
    metric_name_arrs = ['DHS components', 'TF motifs', 'chromatin state themes', 'chromatin state peaks', 'inter-gene associations']
    conversion_dicts = [DHS_conversion_dict, TF_conversion_dict, epi_conversion_dict, epi_conversion_dict, STRING_conversion_dict]

In [19]:
def regional_sentence(rank_arrs, factor_arrs, max_rank_arrs, metric_name_arrs, 
                      conversion_dicts, max_rank=100, multiple_rank_thresh=5, 
                      max_factors_per_metric=3):
    
    num_metrics = len(rank_arrs)
    num_regions = len(rank_arrs[0])

    #establish sentences
    sentences = np.array([''] * num_regions, dtype=object)
    sentence_start = 'This region ranks in the top '
    sentences[:] = sentence_start

    sentence_section_arr = []
    sentence_section_best_ranks = []

    backup_rank = -np.ones(num_regions)
    backup_factor = np.array([''] * num_regions, dtype=object)
    backup_metric = np.array([''] * num_regions, dtype=object)

    #ranks for each metric
    for (rank_arr, factor_arr, 
         max_ranks, metric_name, 
         conversion_dict, metric_i) in zip(rank_arrs, factor_arrs, 
                                           max_rank_arrs, 
                                           metric_name_arrs, 
                                           conversion_dicts,
                                           range(num_metrics)[:-1]):

        #find percent of rank compared to max rank of each factor
        adjusted_ranks = rank_arr / (max_ranks + 1) #pseudocount
        #find the top ranking factor
        rank_sort_order = np.argsort(-adjusted_ranks, axis=1)
        ranks_sorted = -np.sort(-adjusted_ranks, axis=1)

        #int of rank percentage
        #ranks_percent = np.ceil((1 - ranks_sorted) * 100).astype(int)
        ranks_percent = (1 - ranks_sorted) * 100
        ranks_percent = np.where(ranks_percent < 1, np.ceil(ranks_percent * 100) / 100, np.ceil(ranks_percent))
        #set top 0% ranks to top 1% ranks
    #     ranks_percent = np.where(ranks_percent==0, 1, ranks_percent) #not needed with pseudocount

        ###the ranks to include
        #include all values that rank in the top multiple_rank_thresh percent
        ranks_mask = ranks_percent <= multiple_rank_thresh
        #include top factor if rank is in the top max_rank percent
        ranks_mask[:, 0] = ((ranks_mask[:, 0]) | (ranks_percent[:, 0] <= max_rank))
        #number of factors to include for each region 
        num_factors_per_region = np.count_nonzero(ranks_mask, axis=1)

        #find name of top factor
        factors_ranked = factor_arr[rank_sort_order]
        #convert to a readable version of name
        factors_ranked = np.array(list(map(conversion_dict.get, factors_ranked.flatten()))).reshape(factors_ranked.shape)

        #metric in sentence form
        metric_arr = np.char.add(' ', np.array([metric_name] * num_regions))

        #backup for regions
        better_backup_mask = ranks_sorted[:, 0] > backup_rank
        backup_rank[better_backup_mask] = ranks_sorted[:, 0][better_backup_mask]
        backup_factor[better_backup_mask] = factors_ranked[:, 0][better_backup_mask]
        backup_metric[better_backup_mask] = metric_arr[better_backup_mask]

        #initializing the section of the sentence for current metric
        sentence_section = np.array([''] * num_regions)

        metric_ranks_per_region = []
        #for each factor in order of how they rank for each region, add to sentence section
        for factor in range(len(factor_arr))[:max_factors_per_metric]:
            if ((len(ranks_mask[:, factor][ranks_mask[:, factor]]) == 0) & (factor!=0)):
                break
            #mask for factor
            factor_mask = ~ranks_mask[:, factor]
            #percentage
            factor_rank_percent = ranks_percent[:, factor].astype(str)
            factor_rank_percent[factor_mask] = ''

            #saving the rank values for each region
            metric_ranks_per_region.append(np.where(factor_mask, np.nan, ranks_sorted[:, factor]))

            #filler between percent and factor name
            per_of = np.array(['% of '] * num_regions)
            per_of[factor_mask] = ''
            #factor name
            factor_name = factors_ranked[:, factor]
            factor_name[factor_mask] = ''

            #ending
            factor_ending = np.array([''] * num_regions, dtype=object)
            #for regions with more factors than max_factors_per_metric with ranks better than multiple_rank_thresh
            if (factor + 2) == max_factors_per_metric: #if there is 1 factor left
                if factor==0:
                    factor_ending[num_factors_per_region >= (factor + 2)] = ' and ' #>
                else:
                    factor_ending[num_factors_per_region >= (factor + 2)] = ', and ' #>
            elif (factor + 1) == max_factors_per_metric: #if there are no factors left
                factor_ending[num_factors_per_region >= (factor + 2)] = '' #>
            else: #many factors left
                factor_ending[num_factors_per_region > (factor + 2)] = ', '
                if factor==0:
                    factor_ending[num_factors_per_region==(factor + 2)] = ' and '
                else:
                    factor_ending[num_factors_per_region==(factor + 2)] = ', and '

    #         #for regions with 1 more factor left under the multiple_rank_thresh
    #         if (factor + 1) == max_factors_per_metric: #if there are no factors left
    #             factor_ending[num_factors_per_region==(factor + 2)] = ''
    #         else:
    #             factor_ending[num_factors_per_region==(factor + 2)] = ' and '

            #append factor info to sentence section
            sentence_section = np.char.add(sentence_section, factor_rank_percent)
            sentence_section = np.char.add(sentence_section, per_of)
            sentence_section = np.char.add(sentence_section, factor_name)
            sentence_section = np.char.add(sentence_section, factor_ending)

        #add the metric to the sentence section
        metric_arr[~ranks_mask[:, 0]] = ''
        sentence_section = np.char.add(sentence_section, metric_arr)

        sentence_section_arr.append(sentence_section)

        #best ranks for each region
        metric_ranks_per_region = np.array(metric_ranks_per_region).T
        sentence_section_best_ranks.append(np.nanmax(metric_ranks_per_region, axis=1))

    sentence_section_arr = np.array(sentence_section_arr).T
    sentence_section_best_ranks = np.array(sentence_section_best_ranks).T
    sorted_sections = np.argsort(-sentence_section_best_ranks, axis=1)

    #to know where to place '.', ',' and 'and'
    num_sections_with_ranks = np.count_nonzero(~np.isnan(sentence_section_best_ranks), axis=1)

    #add sections to sentences
    for section in range(num_metrics)[:-1]:
        sentences = np.char.add(sentences.astype(str), 
                                sentence_section_arr[np.arange(len(sorted_sections)), 
                                                     sorted_sections[:, section]])
        section_ending = np.array([', '] * num_regions, dtype=object)
        section_ending[num_sections_with_ranks==(section + 1)] = '.'
        section_ending[num_sections_with_ranks<(section + 1)] = ''
        section_ending[(num_sections_with_ranks==(section + 2)) & 
                       (num_sections_with_ranks!=2)] = ', and '
        section_ending[(num_sections_with_ranks==(section + 2)) & 
                       (num_sections_with_ranks==2)] = ' and '
        sentences = np.char.add(sentences.astype(str), 
                                section_ending)

    #fill with backup ranks for regions that don't match filters for any metric
    need_backup_mask = sentences==sentence_start
    #rank
    backup_percent = np.ceil((1 - backup_rank) * 100).astype(int)
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              backup_percent[need_backup_mask].astype(int).astype(str))
    #fill between rank and factor
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              np.array(['% of '] * num_regions)[need_backup_mask])
    #factor
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              backup_factor[need_backup_mask])
    #metric
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              backup_metric[need_backup_mask])
    #.
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              np.array(['.'] * num_regions)[need_backup_mask])
    return sentences

In [131]:
t = time.time()
sentences_lim = regional_sentence_2(rank_arrs, factor_arrs, max_rank_arrs, metric_name_arrs, 
                                    conversion_dicts, max_rank=10, multiple_rank_thresh=3, 
                                    max_factors_per_metric=3)
time.time() - t

/tmp/ipykernel_60274/794047158.py:125: RuntimeWarning: All-NaN slice encountered
  sentence_section_best_ranks.append(np.nanmax(metric_ranks_per_region, axis=1))


0.05167889595031738

In [13]:
sentences_lim[49]

'This region ranks in the top 0.13% of GCM TF motifs, 0.98% of musculoskeletal DHS components, 4.0% of weak transcription chromatin state peaks and 5.0% of weak transcription chromatin state themes.'

In [109]:
sentences_lim[694]

'This region ranks in the top 0.17% of combined inter-gene associations, 0.3% of MAF TF motifs, 3.0% of active TSS chromatin state peaks, 6.0% of stromal B DHS components and 8.0% of upstream flanking TSS chromatin state themes.'

In [134]:
np.argmax(STRING_rank_arr/STRING_max_ranks, axis=0)

array([73001, 18220,   694, 42927,   751, 18220, 11024, 55419])

In [33]:
sentences_lim[78852]

'This region ranks in the top 1.0% of lymphoid and 2.0% of myeloid/erythroid DHS components, 2.0% of HLTF and 2.0% of FOX_8 TF motifs, 5.0% of genic enhancer 2 chromatin state themes, 6.0% of combined inter-gene associations and 6.0% of genic enhancer 2 chromatin state peaks.'

In [42]:
sentences[78852]

'This region ranks in the top 0.01% of lymphoid and 2.0% of myeloid/erythroid DHS components, 2.0% of HLTF and 2.0% of FOX_8 TF motifs, 5.0% of genic enhancer 2 chromatin state themes, 6.0% of combined inter-gene associations, and 6.0% of genic enhancer 2 chromatin state peaks.'

In [53]:
sentences[78852]

'This region ranks in the top 0.01% of lymphoid and 2.0% of myeloid/erythroid DHS components, 2.0% of HLTF and 2.0% of FOX_8 TF motifs, 5.0% of genic enhancer 2 chromatin state themes, and 6.0% of genic enhancer 2 chromatin state peaks.'

In [34]:
sentence_section_best_ranks[78852]

array([1.        , 0.98533156, 0.95693742, 0.94010676, 0.94217646])

In [34]:
f = open(regions_path, 'r')
regions = f.readlines()
f.close()

l = regions[78852].strip().split('\t')
print(l[0] + ':' + l[1] + '-' + l[2])

chr7:50298000-50323000


In [105]:
STRING_rank_arr[73001]

array([53449.])

In [107]:
STRING_max_ranks

array([33181.])

In [167]:
chr3:41173600-41198600

In [112]:
sentences_lim[73001]

'This region ranks in the top 0.12% of BCL6_2 TF motifs, 0.14% of downstream flanking TSS chromatin state themes, 0.72% of downstream flanking TSS chromatin state peaks, 0.76% of combined inter-gene associations and 2.0% of cancer/epithelial DHS components.'

In [159]:
sentences[73001]

'This region ranks in the top 1% of BCL6_2, 1% of NR_4 and 1% of NR_12 TF motifs, 1% of downstream flanking TSS, 1% of genic enhancer 2 and 2% of genic enhancer 1 chromatin state themes, 1% of downstream flanking TSS, 1% of genic enhancer 2 and 2% of active TSS chromatin state peaks, 1% of co-expression and 1% of in vitro gene interassociations and 2% of cancer/epithelial and 3% of lymphoid DHS components.'

In [164]:
sentences_lim[55419]

'This region ranks in the top 1% of co-annotation gene interassociations, 3% of HD_9 TF motifs and 5% of neural DHS components.'

In [160]:
sentences[55419]

'This region ranks in the top 1% of co-annotation, 1% of literature based and 1% of in vitro gene interassociations, 3% of HD_9 and 3% of SCRT1 TF motifs and 5% of neural DHS components.'

In [165]:
sentences_lim[694]

'This region ranks in the top 1% of MAF TF motifs, 1% of co-annotation gene interassociations, 3% of active TSS chromatin state peaks, 6% of stromal B DHS components and 8% of upstream flanking TSS chromatin state themes.'

In [161]:
sentences[694]

'This region ranks in the top 1% of MAF, 1% of ZNF384_2 and 1% of ZNF140 TF motifs, 1% of co-annotation, 1% of literature based and 1% of in vitro gene interassociations, 3% of active TSS chromatin state peaks, 6% of stromal B DHS components and 8% of upstream flanking TSS chromatin state themes.'